In [1]:
import pandas as pd

train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

train_df.columns = ["label", "title", "text"]
test_df.columns = ["label", "title", "text"]

print(train_df.head())

   label                                              title  \
0      3  Wall St. Bears Claw Back Into the Black (Reuters)   
1      3  Carlyle Looks Toward Commercial Aerospace (Reu...   
2      3    Oil and Economy Cloud Stocks' Outlook (Reuters)   
3      3  Iraq Halts Oil Exports from Main Southern Pipe...   
4      3  Oil prices soar to all-time record, posing new...   

                                                text  
0  Reuters - Short-sellers, Wall Street's dwindli...  
1  Reuters - Private investment firm Carlyle Grou...  
2  Reuters - Soaring crude prices plus worries\ab...  
3  Reuters - Authorities have halted oil export\f...  
4  AFP - Tearaway world oil prices, toppling reco...  


In [2]:
print(train_df.dtypes)

print(train_df.shape)
print(test_df.shape)

label     int64
title    object
text     object
dtype: object
(120000, 3)
(7600, 3)


In [3]:
train_df["label"] = train_df["label"].astype(int)
test_df["label"] = test_df["label"].astype(int)

# Input and Output
X_train = train_df["text"].astype(str)
y_train = train_df["label"] - 1

X_test = test_df["text"].astype(str)
y_test = test_df["label"] - 1

print(X_train.head())
print(y_train.head())

0    Reuters - Short-sellers, Wall Street's dwindli...
1    Reuters - Private investment firm Carlyle Grou...
2    Reuters - Soaring crude prices plus worries\ab...
3    Reuters - Authorities have halted oil export\f...
4    AFP - Tearaway world oil prices, toppling reco...
Name: text, dtype: object
0    2
1    2
2    2
3    2
4    2
Name: label, dtype: int64


In [4]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer(num_words=10000)

tokenizer.fit_on_texts(X_train)

X_train = tokenizer.texts_to_sequences(X_train)
X_test = tokenizer.texts_to_sequences(X_test)

In [5]:
max_length = 100

X_train = pad_sequences(X_train, maxlen=max_length)

X_test = pad_sequences(X_test, maxlen=max_length)

In [6]:
from tensorflow.keras.utils import to_categorical

y_train = to_categorical(y_train, 4)
y_test = to_categorical(y_test, 4)

In [7]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

model = Sequential()

model.add(Embedding(input_dim=10000,
                    output_dim=64,
                    input_length=100))

model.add(SimpleRNN(64))

model.add(Dense(32, activation="relu"))

model.add(Dense(4, activation="softmax"))

C:\Users\sandhya\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [8]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [9]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ simple_rnn (SimpleRNN)               │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [10]:
history = model.fit(
    X_train,
    y_train,
    epochs=3,
    batch_size=128,
    validation_split=0.2
)

Epoch 1/3
750/750 ━━━━━━━━━━━━━━━━━━━━ 145s 176ms/step - accuracy: 0.7826 - loss: 0.5543 - val_accuracy: 0.8787 - val_loss: 0.3495
Epoch 2/3
750/750 ━━━━━━━━━━━━━━━━━━━━ 112s 149ms/step - accuracy: 0.9093 - loss: 0.2826 - val_accuracy: 0.8814 - val_loss: 0.3493
Epoch 3/3
750/750 ━━━━━━━━━━━━━━━━━━━━ 141s 147ms/step - accuracy: 0.9341 - loss: 0.2029 - val_accuracy: 0.8676 - val_loss: 0.4045


In [11]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Loss :", loss)
print("Accuracy :", accuracy)

238/238 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - accuracy: 0.8792 - loss: 0.3806
Loss : 0.3806258738040924
Accuracy : 0.8792105317115784


In [12]:
sample = ["India won the cricket world cup"]

sequence = tokenizer.texts_to_sequences(sample)

sequence = pad_sequences(sequence, maxlen=100)

prediction = model.predict(sequence)

classes = [
    "World",
    "Sports",
    "Business",
    "Sci/Tech"
]

print("Predicted Category :", classes[prediction.argmax()])

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step   
Predicted Category : Sports


In [18]:
sample = ["India won the cricket world cup"]

sequence = tokenizer.texts_to_sequences(sample)
sequence = pad_sequences(sequence, maxlen=100)

prediction = model.predict(sequence, verbose=0)

classes = [
    "World",
    "Sports",
    "Business",
    "Sci/Tech"
]

predicted_index = np.argmax(prediction)

print("News :", sample[0])
print("Predicted Category :", classes[predicted_index])
print("Confidence :", prediction[0][predicted_index] * 100, "%")


for i in range(len(classes)):
    print(classes[i], ":", round(prediction[0][i] * 100, 2), "%")

News : India won the cricket world cup
Predicted Category : Sports
Confidence : 99.32393 %
World : 0.63 %
Sports : 99.32 %
Business : 0.02 %
Sci/Tech : 0.02 %


In [17]:
categories = [
    "World",
    "Sports",
    "Business",
    "Science/Technology"
]

text = input("Enter a news article: ")

sequence = tokenizer.texts_to_sequences([text])

sequence = pad_sequences(sequence, maxlen=max_length)

prediction = model.predict(sequence)

index = np.argmax(prediction)

print("Predicted Category:", categories[index])

Enter a news article:  World


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 163ms/step
Predicted Category: Science/Technology
